## Training Stacking Ensemble Learning

<table>
        <thead>
            <tr>
                <th rowspan="2">Modelo</th>
                <th rowspan="2">Acurácia</th>
                <th colspan="2">Precisão</th>
                <th colspan="2">Recall</th>
                <th colspan="2">F1-Score</th>
            </tr>
            <tr>
                <th>Classe 0</th>
                <th>Classe 1</th>
                <th>Classe 0</th>
                <th>Classe 1</th>
                <th>Classe 0</th>
                <th>Classe 1</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>NB - All</td>
                <td>64%</td>
                <td>0.46</td>
                <td>0.83</td>
                <td>0.73</td>
                <td>0.60</td>
                <td>0.56</td>
                <td>0.70</td>
            </tr>
            <tr>
                <td>NB - Selected</td>
                <td>69%</td>
                <td>0.50</td>
                <td>0.82</td>
                <td>0.67</td>
                <td>0.70</td>
                <td>0.58</td>
                <td>0.75</td>
            </tr>
</table>

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

##### Recovering the data

In [14]:
X_train = pd.read_csv("../../datasets/okx_datasets/X_train.csv")
X_test  = pd.read_csv("../../datasets/okx_datasets/X_test.csv")
y_train = pd.read_csv("../../datasets/okx_datasets/y_train.csv").values.ravel()
y_test  = pd.read_csv("../../datasets/okx_datasets/y_test.csv").values.ravel()

In [15]:
print("Shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Shapes:
X_train: (180, 192)
X_test: (46, 192)
y_train: (180,)
y_test: (46,)


#### Training the model with all variables

In [16]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# pipe = Pipeline([
#     ("stack", StackingClassifier(
#         estimators=[
#             ("rf", RandomForestClassifier(random_state=42, n_jobs=-1)),
#             ("et", ExtraTreesClassifier(random_state=42, n_jobs=-1)),
#             ("gb", GradientBoostingClassifier(random_state=42)),
#             ("knn", KNeighborsClassifier())
#         ],
#         final_estimator=LogisticRegression(max_iter=3000),
#         cv=5,
#         n_jobs=-1,
#         passthrough=False
#     ))
# ])

# param_grid = {
#     "stack__rf__n_estimators": [100, 200],
#     "stack__et__n_estimators": [100, 200],
#     "stack__gb__n_estimators": [50, 100],
#     "stack__final_estimator__C": [0.1, 1.0, 10.0]
# }

# grid = GridSearchCV(
#     estimator=pipe,
#     param_grid=param_grid,
#     cv=5,
#     scoring="accuracy",
#     n_jobs=-1
# )

# grid.fit(X_train, y_train)

# print("Melhores hiperparâmetros encontrados:")
# print(grid.best_params_)
# print("\nMelhor score de validação:", grid.best_score_)

# best_model = grid.best_estimator_
# y_pred = best_model.predict(X_test)

# acc = accuracy_score(y_test, y_pred)
# prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
# rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
# f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

# print("\n==================== MÉTRICAS MODELO FINAL ====================")
# print(f"Acurácia:   {acc:.4f}")
# print(f"Precisão:   {prec:.4f}")
# print(f"Recall:     {rec:.4f}")
# print(f"F1-score:   {f1:.4f}")
# print("\n==================== CLASSIFICATION REPORT ====================")
# print(classification_report(y_test, y_pred, zero_division=0))

In [17]:
pipe = Pipeline([
    ("select", SelectKBest(score_func=f_classif)),
    ("stack", StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(random_state=42, n_jobs=-1)),
            ("et", ExtraTreesClassifier(random_state=42, n_jobs=-1)),
            ("gb", GradientBoostingClassifier(random_state=42)),
            ("knn", KNeighborsClassifier())
        ],
        final_estimator=LogisticRegression(max_iter=3000),
        cv=5,
        n_jobs=-1,
        passthrough=False
    ))
])

param_grid = {
    "select__k": [5, 10, 15, "all"],
    "stack__rf__n_estimators": [100, 200],
    "stack__et__n_estimators": [100, 200],
    "stack__gb__n_estimators": [50, 100],
    "stack__final_estimator__C": [0.1, 1.0, 10.0]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Melhores hiperparâmetros encontrados:")
print(grid.best_params_)
print("\nMelhor score de validação:", grid.best_score_)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

selector = best_model.named_steps["select"]
selected_features = X_train.columns[selector.get_support()]
print("\nFeatures selecionadas:")
print(list(selected_features))

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("\n==================== MÉTRICAS MODELO FINAL ====================")
print(f"Acurácia:   {acc:.4f}")
print(f"Precisão:   {prec:.4f}")
print(f"Recall:     {rec:.4f}")
print(f"F1-score:   {f1:.4f}")
print("\n==================== CLASSIFICATION REPORT ====================")
print(classification_report(y_test, y_pred, zero_division=0))

Melhores hiperparâmetros encontrados:
{'select__k': 5, 'stack__et__n_estimators': 100, 'stack__final_estimator__C': 0.1, 'stack__gb__n_estimators': 50, 'stack__rf__n_estimators': 100}

Melhor score de validação: 1.0

Features selecionadas:
['Age', 'T-score Value', 'Z-Score Value', 'Occupation _student', 'Medical History_uterus rem, appendex, disk']

==================== MÉTRICAS MODELO FINAL ====================
Acurácia:   0.9783
Precisão:   0.9790
Recall:     0.9783
F1-score:   0.9776

==================== CLASSIFICATION REPORT ====================
              precision    recall  f1-score   support

           0       1.00      0.86      0.92         7
           1       0.97      1.00      0.98        30
           2       1.00      1.00      1.00         9

    accuracy                           0.98        46
   macro avg       0.99      0.95      0.97        46
weighted avg       0.98      0.98      0.98        46



In [18]:
import joblib

stacking_model = best_model.named_steps["stack"]
joblib.dump(stacking_model, "stacking_model.pkl")
print("Modelo salvo com sucesso usando apenas as 5 variáveis selecionadas!")

Modelo salvo com sucesso usando apenas as 5 variáveis selecionadas!
